# alarm_granularity_sweep

Two ways to build unfavourable burden (`pUF` = topics 3+4+5):

1. **Simple / sliding** — each day `d≥30`, LDA on the previous 30 days → `pUF(d)`. Then evaluate daily or subsample every 7 / 14 / 30 days.
2. **Collapsed** — non-overlapping blocks of 7 / 14 / 30 days; one LDA mixture per block.

**W** = lookback (days of burden before the decision). **H** = prediction horizon (days ahead for PD).

Paper reference: monthly collapsed, W≈90 d (3 mo), H≈120 d (4 mo), θ=1.61.

```bash
.venv/bin/python scripts/03_analysis/alarm/run_granularity_sweep.py --skip-lda
.venv/bin/python scripts/03_analysis/alarm/run_granularity_sweep.py --monthly-only
```


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import auc, roc_curve

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "scripts/03_analysis/alarm"))

from rolling_burden_alarm import extract_monthly_puf_from_master, default_master_table_path
from sliding_burden_alarm import (
    build_sliding_decision_points,
    build_collapsed_decision_points,
    build_monthly_collapsed_decision_points,
)

TABLES = ROOT / "results/alarm/tables"
FIGS = ROOT / "results/alarm/figures"


## 1 · Load precomputed outputs

In [ ]:
daily_puf = pd.read_csv(TABLES / "daily_sliding_puf.csv")
summary = pd.read_csv(TABLES / "granularity_sweep_summary.csv")
print(f"Daily pUF rows: {len(daily_puf):,} | patients: {daily_puf['id'].nunique()}")
summary.sort_values(["granularity", "w_load_days", "h_horizon_days"]).head(10)

## 2 · Paper reference row (monthly collapsed, W≈90 d, H≈120 d)

In [ ]:
ref = summary[(summary.w_load_days == 90) & (summary.h_horizon_days == 120)].copy()
ref.sort_values("granularity")[
    ["granularity", "n_decision_points", "sensitivity", "specificity", "PPV", "NPV", "roc_auc", "youden_threshold"]
]


## 3 · ROC curves by granularity (W=90 d, H=120 d)

In [ ]:
W, H = 90, 120
master = pd.read_excel(default_master_table_path(ROOT))
monthly_puf = extract_monthly_puf_from_master(master, root=ROOT)

fig, ax = plt.subplots(figsize=(7.2, 6))
curves = [
    ("daily", build_sliding_decision_points(daily_puf, "daily", W, H)),
    ("weekly sampled", build_sliding_decision_points(daily_puf, "weekly", W, H)),
    ("biweekly sampled", build_sliding_decision_points(daily_puf, "biweekly", W, H)),
    ("monthly sampled", build_sliding_decision_points(daily_puf, "monthly_sample", W, H)),
]
for name, block, path in [
    ("weekly collapsed", 7, TABLES / "collapsed_puf_weekly.csv"),
    ("biweekly collapsed", 14, TABLES / "collapsed_puf_biweekly.csv"),
]:
    if path.exists():
        curves.append(
            (name, build_collapsed_decision_points(pd.read_csv(path), block, W, H, granularity=name))
        )
curves.append(
    ("monthly collapsed", build_monthly_collapsed_decision_points(monthly_puf, w_months=3, h_months=4))
)

for label, dp in curves:
    fpr, tpr, _ = roc_curve(dp["target"], dp["burden_score"])
    ax.plot(fpr, tpr, lw=2, label=f"{label} (AUC={auc(fpr, tpr):.2f}, n={len(dp)})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("FPR (1 – specificity)")
ax.set_ylabel("TPR (sensitivity)")
ax.set_title(f"Granularity comparison (W={W} d, H={H} d)")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## 4 · Heatmap: ROC-AUC by granularity × (W, H)

In [ ]:
pivot = summary.pivot_table(
    index="granularity",
    columns=["w_load_days", "h_horizon_days"],
    values="roc_auc",
)
pivot